# [8.2] Attribution Patching and EAP - Exercises

Exact activation patching is the reference experiment: patch a clean activation into a corrupt run and measure the recovered logit-diff. Attribution patching asks whether a gradient can predict which exact patches are worth running first.

This notebook starts with toy tensors because the contracts are easier to debug there. By the end, you should have the same objects used by the pinned CUDA report: exact scores, corrupt-run attribution scores, integrated-gradient scores, an EAP-style edge matrix, top-k agreement, and explicit false-negative documentation.

```text
clean prompt:   The cat sat on the      target token: " floor"
corrupt prompt: The bird flew over the  distractor:    " top"
accepted exact scores by position:       [0, 0, 0, 0, 0, 1.0]
accepted attribution scores by position: [0, 0, 0, 0, 0, 0.9608]
accepted IG scores by position:          [0, 0, 0, 0, 0, 0.9732]
```

A section-ready result is not just green tests. You should be able to say why exact patching is still the causal reference, why attribution/EAP are prioritization tools, and why a false negative matters more than a pretty average correlation.

<details>
<summary>Expected output</summary>

By the end of the notebook, your CPU smoke report should contain attribution and integrated-gradient scores `[1.0, 3.0]`, edge scores `[[3.0, 0.0], [0.0, 8.0]]`, correlation about `0.9966`, top-k overlap `1.0`, runtime speedup `5.0`, and one documented false negative at index `1`. The committed CUDA report should show best position `5` for exact, attribution, and IG scores, with EAP top edge `(5, 5)`.

</details>

<details>
<summary>Help - how this mirrors original ARENA patching</summary>

Original ARENA moves from direct logit attribution to activation patching and then path patching. This section keeps that rhythm: exact node patching is the reference, gradient attribution is the cheap approximation, and EAP is the edge-shaped approximation to a path-patching question.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch as t

chapter = "chapter8_automated_circuits"
section = "part2_attribution_patching_eap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_attribution_patching_eap.tests as tests
import part2_attribution_patching_eap.utils as utils

GT_TIER = "GT-1"
EXERCISE_ID = "8_2_attribution_patching_and_eap"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True


## First-Order Attribution Patching

Attribution patching estimates the patch effect with a first-order Taylor approximation:

```text
patch effect ~= (clean activation - corrupt activation) dot corrupt gradient
```

The function should preserve the component axis and reduce every other axis. For activations shaped `[position, d_model]`, return one score per position. For activations shaped `[batch, position, d_model]` with `component_dim=1`, return one score per position after summing batch and hidden dimensions.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_attribution_patch_scores_sums_non_component_dims` passed!
All tests in `test_attribution_patch_scores_rejects_degenerate_inputs` passed!
```

The toy clean/corrupt/gradient tensors should produce attribution scores `[1.0, 3.0]`.

</details>

<details>
<summary>Help - where does the formula come from?</summary>

The corrupt activation is the point where you know the gradient of the metric. The clean-corrupt activation delta is the direction you want to move. Their dot product is the first-order estimate of how much the metric would change if you moved from corrupt toward clean.

</details>

<details>
<summary>Common bug</summary>

Do not sum over the component dimension. If you return a scalar, you have lost the ranking signal that tells you which component to inspect.

</details>

<details>
<summary>Solution sketch</summary>

Validate matching shapes, validate `component_dim`, reject non-finite tensors, compute `(clean.float() - corrupt.float()) * gradients.float()`, and sum over every axis except `component_dim`.

</details>


In [ ]:
def attribution_patch_scores(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    corrupt_gradients: t.Tensor,
    *,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_attribution_patch_scores_sums_non_component_dims(
    attribution_patch_scores,
)
tests.test_attribution_patch_scores_rejects_degenerate_inputs(
    attribution_patch_scores,
)


## Integrated-Gradient Patching

Corrupt-run attribution uses one gradient. Integrated-gradient patching averages gradients along a path from corrupt activations to clean activations. This can be less brittle when the metric changes nonlinearly along the path, but it is still an approximation.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_integrated_gradient_patch_scores_average_path_gradients` passed!
All tests in `test_integrated_gradient_patch_scores_rejects_empty_or_nonfinite_paths` passed!
```

The toy path gradients should also produce scores `[1.0, 3.0]`, because their mean gradient equals the corrupt-run gradient in the fixture.

</details>

<details>
<summary>Help - what does the leading step dimension mean?</summary>

`path_gradients` has shape `[steps, *activation_shape]`. Each step is one gradient evaluated at an interpolation point between corrupt and clean activations. Average over `steps`, then use the same attribution-patching formula.

</details>

<details>
<summary>Common bug</summary>

Do not sum the path gradients. Integrated gradients average over the path; summing makes the score grow just because you used more interpolation points.

</details>

<details>
<summary>Solution sketch</summary>

Require `path_gradients.ndim == clean_activations.ndim + 1`, reject empty or non-finite paths, average over the leading step dimension, then call `attribution_patch_scores` with the mean gradient.

</details>


In [ ]:
def integrated_gradient_patch_scores(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    path_gradients: t.Tensor,
    *,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_integrated_gradient_patch_scores_average_path_gradients(
    integrated_gradient_patch_scores,
)
tests.test_integrated_gradient_patch_scores_rejects_empty_or_nonfinite_paths(
    integrated_gradient_patch_scores,
)


## Edge Attribution Patching

Activation patching scores nodes. Path patching asks edge questions. EAP is the gradient approximation to that edge question: pair each upstream activation delta with each downstream gradient and form an upstream-by-downstream matrix.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_edge_attribution_scores_forms_upstream_downstream_matrix` passed!
All tests in `test_edge_attribution_scores_rejects_empty_or_nonfinite_inputs` passed!
```

The toy upstream/downstream matrices should produce:

```text
[[3.0, 0.0],
 [0.0, 8.0]]
```

</details>

<details>
<summary>Help - node scores vs edge scores</summary>

Attribution patching gives one score per component. EAP gives one score per ordered pair of components: upstream sender and downstream receiver. This is closer to path patching, but it is still a gradient approximation rather than an intervention.

</details>

<details>
<summary>Common bug</summary>

Elementwise multiplying only matching component ids loses cross edges. You want matrix multiplication: `upstream_delta @ downstream_gradients.T`.

</details>

<details>
<summary>Solution sketch</summary>

Validate both inputs have shape `(components, d_model)`, reject empty component or hidden dimensions, reject non-finite tensors, and return `upstream_activation_delta.float() @ downstream_gradients.float().T`.

</details>


In [ ]:
def edge_attribution_scores(
    upstream_activation_delta: t.Tensor,
    downstream_gradients: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_edge_attribution_scores_forms_upstream_downstream_matrix(
    edge_attribution_scores,
)
tests.test_edge_attribution_scores_rejects_empty_or_nonfinite_inputs(
    edge_attribution_scores,
)


## Exact-vs-Approx Accountability

Correlation checks global agreement with exact patching. Top-k overlap checks whether the approximate method points you to inspect the same most important components. You need both: a method can have high average agreement while still missing the one component you cared about.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_exact_vs_approx_reports_measure_correlation_and_topk_overlap` passed!
All tests in `test_exact_vs_approx_reports_reject_bad_thresholds_and_scores` passed!
```

For exact scores `[0.1, 0.9, 0.8, 0.0]` and approximate scores `[0.2, 0.85, 0.7, 0.1]`, correlation should be about `0.9966` and top-2 overlap should be `1.0`.

</details>

<details>
<summary>Help - why not just use correlation?</summary>

Circuit work often depends on the top few components. Top-k overlap makes the question explicit: if I only have time to run exact interventions on two things, did the approximation point me to the same two things?

</details>

<details>
<summary>Common bug</summary>

Cosine similarity is not Pearson correlation. Center both score vectors before taking the normalized dot product.

</details>

<details>
<summary>Solution sketch</summary>

Flatten both tensors, reject mismatched shapes and non-finite values, compute centered Pearson correlation, then compute top-k overlap from the sets of largest exact and approximate indices.

</details>


In [ ]:
@dataclass(frozen=True)
class ScoreCorrelationReport:
    correlation: float
    passes_threshold: bool


@dataclass(frozen=True)
class TopKOverlapReport:
    exact_top_indices: tuple[int, ...]
    approx_top_indices: tuple[int, ...]
    topk_overlap: float
    passes_threshold: bool


def score_correlation_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    min_correlation: float = 0.8,
) -> ScoreCorrelationReport:
    raise NotImplementedError()


def topk_overlap_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    top_k: int = 3,
    min_overlap: float = 0.5,
) -> TopKOverlapReport:
    raise NotImplementedError()


tests.test_exact_vs_approx_reports_measure_correlation_and_topk_overlap(
    score_correlation_report,
    topk_overlap_report,
)
tests.test_exact_vs_approx_reports_reject_bad_thresholds_and_scores(
    score_correlation_report,
    topk_overlap_report,
)


## Runtime and False Negatives

Approximate patching earns its place with speed, but speed is not enough. It must also document exact-important components that the approximation misses. This is the part that prevents "fast and wrong" from looking like a success.

> ```yaml
> Difficulty: medium
> Importance: high
> You should spend 10-15 minutes on this exercise.
> ```

<details>
<summary>Expected output</summary>

```text
All tests in `test_runtime_and_false_negative_reports_enforce_accountability` passed!
All tests in `test_runtime_and_false_negative_reports_reject_nonfinite_inputs` passed!
```

The toy runtime report should show `speedup = 5.0`. The toy false-negative report should identify index `1` and mark it documented only if the note is nonempty.

</details>

<details>
<summary>Help - false negatives are not optional bookkeeping</summary>

If exact patching says a component matters and the approximation misses it, that is the most important line in the report. A good attribution workflow makes misses visible so you know when to fall back to exact patching.

</details>

<details>
<summary>Common bug</summary>

The speedup is `exact_runtime_s / approx_runtime_s`. Reversing the ratio rewards slower approximations.

</details>

<details>
<summary>Solution sketch</summary>

Validate runtimes and thresholds, compute speedup, then find indices where `exact >= exact_threshold` but `approx < approx_threshold`. The report is documented only when every missed index has a nonempty note.

</details>


In [ ]:
@dataclass(frozen=True)
class RuntimeImprovementReport:
    exact_runtime_s: float
    approx_runtime_s: float
    speedup: float
    passes_speedup: bool


@dataclass(frozen=True)
class FalseNegativeReport:
    false_negative_indices: tuple[int, ...]
    num_false_negatives: int
    documented: bool


def runtime_improvement_report(
    *,
    exact_runtime_s: float,
    approx_runtime_s: float,
    min_speedup: float = 2.0,
) -> RuntimeImprovementReport:
    raise NotImplementedError()


def false_negative_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    exact_threshold: float,
    approx_threshold: float,
    documentation: dict[int, str] | None = None,
) -> FalseNegativeReport:
    raise NotImplementedError()


tests.test_runtime_and_false_negative_reports_enforce_accountability(
    runtime_improvement_report,
    false_negative_report,
)
tests.test_runtime_and_false_negative_reports_reject_nonfinite_inputs(
    runtime_improvement_report,
    false_negative_report,
)


## Combined Contract

Compose the local helpers into a CPU-only smoke report before touching the live model. The smoke report should have the same semantic pieces as the CUDA report: attribution scores, IG scores, edge scores, agreement checks, runtime, and false-negative accountability.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

The contract should expose attribution and IG scores `[1.0, 3.0]`, edge scores `[[3.0, 0.0], [0.0, 8.0]]`, correlation about `0.9966`, top-k overlap `1.0`, speedup `5.0`, and a documented false negative at index `1`.

</details>

<details>
<summary>Help - reading the combined contract</summary>

The smoke report is not model evidence. It is a debugging contract that proves your implementation has the same objects the model path will need.

</details>

<details>
<summary>Solution sketch</summary>

Use the toy tensors from the previous exercises, convert dataclasses to dictionaries, and return a JSON-like report with one key for each contract.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)


## Signature Result

The committed CUDA report is the section-scale result. It should show a pinned `gelu-1l` TransformerLens run where exact residual-stream patching, corrupt-run attribution patching, integrated-gradient patching, and the EAP position-edge matrix all point to the final residual position.

<details>
<summary>Expected output</summary>

```text
preflight_passed: true
exact_best_position: 5
attribution_best_position: 5
ig_best_position: 5
exact_final_recovery: 1.0
attribution_final_recovery: 0.9608...
ig_final_recovery: 0.9732...
eap_top_edge: (5, 5)
within_vram_budget: true
```

</details>

<details>
<summary>Help - what does this result prove?</summary>

It proves that the local attribution/EAP mechanics recover the same final residual-position target as exact patching on one pinned safe prompt pair. It does not prove IOI-scale EAP, dataset-level faithfulness, or a discovered circuit.

</details>

<details>
<summary>Common bug</summary>

Do not describe this as replacing exact patching. Approximate methods triage what to patch exactly next.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["preflight_passed"]
    assert gpu["exact_best_position"] == gpu["target_position"] == 5
    assert gpu["attribution_best_position"] == 5
    assert gpu["ig_best_position"] == 5
    assert gpu["exact_final_recovery"] >= 0.99
    assert gpu["attribution_final_recovery"] >= 0.9
    assert gpu["ig_final_recovery"] >= 0.95
    assert gpu["exact_attribution_top1_overlap"] == 1.0
    assert gpu["exact_ig_top1_overlap"] == 1.0
    assert gpu["eap_top_edge_upstream_position"] == 5
    assert gpu["eap_top_edge_downstream_position"] == 5
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test(max_vram_gb=24.0)
utils.print_report(
    "Committed CUDA exact-vs-attribution report",
    {
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "device": gpu["device"],
        "exact": gpu["exact_patch_scores_by_position"],
        "attribution": [round(x, 4) for x in gpu["attribution_scores_by_position"]],
        "ig": [round(x, 4) for x in gpu["ig_scores_by_position"]],
        "eap_top_edge": (
            gpu["eap_top_edge_upstream_position"],
            gpu["eap_top_edge_downstream_position"],
        ),
        "peak_vram_gb": round(gpu["peak_vram_gb"], 4),
    },
)


## Limitations

This is a GT-1 attribution/EAP mechanics preflight on one pinned `gelu-1l` hook and one safe prompt pair. It is not IOI-scale EAP replication, not a full path-patching intervention suite, not a dataset-level causal graph, and not evidence that attribution patching is reliable without exact-patching checks.

## Further Research

Repeat exact-vs-approx comparisons across prompt templates, move from residual positions to heads/MLPs/SAE features, compare EAP edges against true path-patching interventions, and add false-negative case studies where high global correlation still misses an exact-important component.
